# Tutorial: `rust-scHiCluster` quickstart

This notebook walks through the three things you'll do with
`rust_schicluster`:

1. **Import + check** the Rust extension built correctly.
2. **Drop-in monkey-patch** — replace upstream `schicluster.impute_chromosome`
   transparently. All your existing scHiCluster CLI / snakemake / downstream
   tooling gets the speed-up with zero code changes.
3. **Multi-process parallelism** — `set_num_threads()` to avoid rayon
   thread oversubscription when running across many cells in
   `ProcessPoolExecutor`.

Build the extension first:

```bash
git clone https://github.com/omicverse/rust-scHiCluster
cd rust-scHiCluster
maturin develop --release
```

## 1. Import and verify

In [1]:
import rust_schicluster

print('rust extension available:', rust_schicluster._RUST_AVAILABLE)
print('public API:', rust_schicluster.__all__)

rust extension available: True
public API: ['random_walk_cpu', 'impute_chromosome', 'patch_schicluster', 'set_num_threads']


## 2. Direct use — `random_walk_cpu` on a CSR matrix

Same signature as upstream
`schicluster.impute.impute_chromosome.random_walk_cpu`:
input is a `scipy.sparse.csr_matrix` (or anything `csr_matrix(...)`
accepts), output is the converged Q as CSR.

In [2]:
import numpy as np
from scipy.sparse import csr_matrix, diags

# Tiny example — 100×100 row-stochastic sparse matrix.
rng = np.random.default_rng(0)
nnz = 500
rows = rng.integers(0, 100, size=nnz)
cols = rng.integers(0, 100, size=nnz)
vals = rng.exponential(1.0, size=nnz).astype(np.float32)
M = csr_matrix((vals, (rows, cols)), shape=(100, 100), dtype=np.float32)
M = M + M.T
row_sum = np.asarray(M.sum(axis=1)).ravel()
row_sum[row_sum == 0] = 1.0
P = diags(1.0 / row_sum) @ M

Q = rust_schicluster.random_walk_cpu(P, rp=0.5, tol=0.01)
print(f'Q: {Q.shape}, nnz={Q.nnz}, dtype={Q.dtype}')
print(f'sum check: 1 row sums to ≈ 1.0  →  {np.asarray(Q.sum(axis=1))[:3].ravel()}')

Q: (100, 100), nnz=10000, dtype=float32
sum check: 1 row sums to ≈ 1.0  →  [0.99999994 1.         1.0000001 ]


## 3. Drop-in monkey-patch

`patch_schicluster()` rebinds two symbols in upstream:

* `schicluster.impute.impute_chromosome.random_walk_cpu` → Rust kernel
* `schicluster.impute.impute_chromosome.impute_chromosome` → Rust full
  pipeline (Gaussian conv + RWR + SQRTVC + triangle filter)

All downstream code (CLI tools, snakemake, packages like `epione.single.hic`)
that imports from `schicluster.impute.impute_chromosome` then sees the
Rust version. Returns `True` on success, `False` if the extension
wasn't compiled (graceful fallback to upstream Python).

In [3]:
import schicluster.impute.impute_chromosome as _mod

before = _mod.impute_chromosome
ok = rust_schicluster.patch_schicluster()
after = _mod.impute_chromosome

print(f'patch ok: {ok}')
print(f'before: {before.__module__}.{before.__qualname__}')
print(f'after : {after.__module__}.{after.__qualname__}')

patch ok: True
before: schicluster.impute.impute_chromosome.impute_chromosome
after : rust_schicluster.impute_chromosome


## 4. Multi-process parallelism — `set_num_threads()`

The biggest pitfall when running impute on many cells with
`ProcessPoolExecutor`: each worker forks the rayon pool with
`num_cpus` threads by default. With 8 workers on a 16-core node you
get 8×16 = 128 contending threads — slower than running each cell
single-threaded.

Recommended sizing: `n = num_cpus // num_workers`. Example for a
16-core node with 8 workers → `set_num_threads(2)`.

In [4]:
from concurrent.futures import ProcessPoolExecutor

def worker_init():
    """Run once per worker process, before any impute call."""
    import rust_schicluster
    rust_schicluster.set_num_threads(2)        # 8 workers × 2 = 16
    rust_schicluster.patch_schicluster()        # Rust kernel everywhere

def impute_one_cell(args):
    cool_path, chrom, output_path = args
    from schicluster.impute.impute_chromosome import impute_chromosome
    impute_chromosome(
        scool_url=cool_path, chrom=chrom, resolution=25_000,
        output_path=output_path,
        rp=0.5, tol=0.01, pad=1, std=1.0,
        window_size=10_000_000_000, output_dist=10_050_000,
    )

# Example skeleton (don't actually run — needs real .cool paths):
#
# args_list = [(cool, chrom, out) for cool, chrom, out in your_cells]
# with ProcessPoolExecutor(max_workers=8, initializer=worker_init) as ex:
#     list(ex.map(impute_one_cell, args_list))

## 5. Math approximation — `band_factor` for extra speed

If you can tolerate ~1% deviation from the strict scipy result,
`impute_chromosome(..., band_factor=N)` runs the RWR with a banded Q
(only entries with `|j − i| ≤ N · output_dist_bins`), giving an
additional ~4× speed-up.

Math justification: out-of-band signal in RWR decays geometrically as
`(1 − rp)^d` per bin distance, so for `rp = 0.5` and a band of
`4 × output_dist_bins` the dropped signal is bounded by `0.5^(4·B)` —
machine zero for any practical `B ≥ 100`.

Default is `band_factor=0` (off, strict bit-equivalent). Use this
approximation only for production batch processing where ~1% absolute
value drift on individual pixels is acceptable; structure-preserving
downstream analyses (compartments, TADs, scGAD, UMAP) are unaffected.

In [5]:
# Skeleton (replace COOL):
# rust_schicluster.impute_chromosome(
#     scool_url=COOL, chrom='chr1', resolution=25_000,
#     output_path='/tmp/chr1_fast.hdf',
#     rp=0.5, tol=0.01, pad=1, std=1.0, output_dist=10_050_000,
#     band_factor=4,                    # ← the only change
# )
print('see notebook above for inline use; band_factor=4 gives ~4× extra')

see notebook above for inline use; band_factor=4 gives ~4× extra


## 6. Where to go next

* `examples/benchmark_vs_scipy.ipynb` — bit-equivalence test + speed table
* `tests/test_parity.py` — automated parity assertions (run with `pytest`)
* The downstream package [`epione`](https://github.com/aristoteleo/epione) uses `rust-scHiCluster` for sc-Hi-C imputation in `epione.single.hic`.